# 예제 03: 개별 조인트 제어 (self-contained)

각 조인트를 하나씩 움직여 어떤 조인트가 어떤 동작을 하는지 시각적으로 확인하는 예제.
`utils.py`를 사용하지 않고 단일 노트북으로 완결되도록 구성했다.

**학습 내용**
- `MoveGroup` 액션과 `MotionPlanRequest` / `JointConstraint` 메시지 구성
- `velocity_scaling_factor` / `acceleration_scaling_factor` 로 속도 조절
- 각 조인트(joint1~joint6)의 역할 직관적으로 파악

원본 스크립트: `ex03_joint_goal.py`

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz의 `move_group` 액션 서버에 클라이언트로 붙는 방식이다.
터미널을 둘 띄워야 한다.

> ⚠ 다른 로봇용 MoveIt launch (`franka_with_d435.launch.py` 등)가 떠 있으면 같은 토픽으로 충돌해 RViz가 죽거나 controller_manager가 segfault할 수 있다. 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch robot_arm_moveit_config demo.launch.xml
```

RViz 창이 뜨고 `MotionPlanning` 패널이 보이면 준비 완료.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate     # ros_jazzy venv 활성화 (jupyter 설치된 곳)
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab ex03_joint_goal.ipynb         # 또는 jupyter notebook
```

브라우저가 이 노트북을 열면, 아래 셀들을 위에서 아래로 순서대로 실행한다
(`Shift+Enter` 한 번이면 셀 실행 + 다음 셀로 이동).

### 중간에 멈추거나 다시 실행하고 싶을 때

- 마지막 "정리" 셀까지 가지 않고 닫아도 된다. 다음에 열 때 *반드시 위쪽 셀부터 다시 실행*해야 `node`, `move_client`, `home_target` 같은 변수가 살아 있다.
- `rclpy`는 한 프로세스에서 한 번만 init할 수 있어, **2-2. `rclpy` 초기화** 셀은 `try/except`로 감싸 두 번째 실행해도 무시한다.
- 노트북을 재시작하지 않고 처음부터 다시 시뮬레이션하려면 마지막 "정리" 셀을 실행한 뒤, 위에서부터 다시 돈다.

### `use_sim_time` 관련 참고

`demo.launch.xml`의 `move_group`이 sim time을 쓰지 않으므로 이 노트북도 따로 `use_sim_time` 파라미터를 설정하지 않는다.
Gazebo 같은 시뮬레이터와 함께 돌릴 때는 노드 생성 시 `parameter_overrides=[Parameter('use_sim_time', value=True)]` 를 추가해야 시간 동기가 맞는다.

## 1. 로봇 상수 정의

이 값들은 `robot_arm_moveit_config/config/robot_arm.srdf`와 일치해야 한다.

In [1]:
PLANNING_GROUP   = 'manipulator'
REFERENCE_FRAME  = 'base_link'
ARM_JOINTS       = ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6']

## 2. ROS 2 초기화와 노드 생성

이 섹션에서 처음 쓰이는 `rclpy`, `Node`, `ActionClient`, `JointState`, `MoveGroup` 을 import한 뒤,
`rclpy.init()` → 노드/액션 클라이언트/구독자 생성 순서로 진행한다.

### 2-1. import

In [2]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup

### 2-2. `rclpy` 초기화

한 프로세스에서 한 번만 init 가능하므로, 노트북에서 이 셀을 두 번 실행해도 무시되도록 `try/except`로 감싼다.

In [3]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우(노트북에서 재실행)는 무시

### 2-3. 노드 + 액션 클라이언트 + `joint_states` 구독자

`MoveGroup` 액션 클라이언트와 현재 관절 상태를 받는 구독자를 같은 노드에 붙인다.

In [4]:
node = Node('ex03_joint_goal_demo')
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== 예제 03 노트북 노드 생성 완료 ===')

[INFO] [1778197588.366555470] [ex03_joint_goal_demo]: === 예제 03 노트북 노드 생성 완료 ===


True

## 3. 액션 서버와 `/joint_states` 준비 대기

현재 관절 위치를 한 번 이상 받아본 뒤에 모션 플래닝 요청을 보내야 안정적이다.

In [5]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

[INFO] [1778197637.260643026] [ex03_joint_goal_demo]: action server + /joint_states 준비됨


## 4. SRDF에서 `home` 포즈 읽어오기 — 기능별로 분리한 헬퍼들

`move_group` 노드가 보유한 `robot_description_semantic` 파라미터에서 SRDF XML을 받아
`<group_state name="home" group="manipulator">` 안의 조인트 값들을 딕셔너리로 추출한다.
한 함수에 다 넣지 않고 *XML 가져오기*와 *파싱*을 분리하면 각각 따로 테스트할 수 있다.

| 함수 | 역할 |
|---|---|
| `fetch_srdf_xml` | `move_group`에서 `robot_description_semantic` 파라미터(SRDF XML 문자열)를 받아옴 |
| `parse_named_pose` | SRDF XML에서 지정 group/name 의 group_state 를 dict로 파싱 |
| `load_named_pose` | 위 둘을 묶는 얇은 오케스트레이터 |

### 4-1. `move_group`에서 SRDF XML 가져오기

In [6]:
from rclpy.parameter_client import AsyncParameterClient

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

### 4-2. SRDF XML에서 group_state 파싱

In [7]:
import xml.etree.ElementTree as ET

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

### 4-3. 두 단계를 묶는 진입점

In [8]:
def load_named_pose(name: str = 'home', timeout_sec: float = 10.0) -> dict:
    srdf_xml = fetch_srdf_xml(timeout_sec)
    return parse_named_pose(srdf_xml, name, PLANNING_GROUP)

### 4-4. `home` 포즈 가져와 변수에 저장

In [9]:
home_target = load_named_pose('home')
node.get_logger().info(f'home target: {home_target}')

[INFO] [1778197771.594087136] [ex03_joint_goal_demo]: home target: {'joint1': 0.0, 'joint2': 0.0, 'joint3': 0.0, 'joint4': 0.0, 'joint5': 0.0, 'joint6': 0.0}


True

## 5. MoveGroup 목표 전송 — 기능별로 분리한 헬퍼들

원래 한 함수에 다 들어 있던 동작을 5개의 작은 헬퍼와 1개의 오케스트레이터로 나눴다.
각각 하나의 일만 하므로 디버깅·재사용·교체가 쉽다.

| 함수 | 역할 |
|---|---|
| `make_joint_constraints` | 조인트 dict → `Constraints` 메시지 |
| `make_plan_request` | 위 둘을 합쳐 완성된 `MotionPlanRequest` 생성 |
| `make_goal` | `MotionPlanRequest`를 `MoveGroup.Goal`로 감싸기 (planning options 포함) |
| `send_goal_and_wait` | 액션 send + accept + result 대기, error code 반환 |
| `go_to_joint_goal` | 위 함수들을 호출하는 최종 진입점 |

### 5-2. 조인트 제약 만들기

In [10]:
from moveit_msgs.msg import Constraints, JointConstraint

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    constraints = Constraints()
    for jname, val in joint_values.items():
        jc = JointConstraint(joint_name=jname, position=val,
                              tolerance_above=tol, tolerance_below=tol,
                              weight=1.0)
        constraints.joint_constraints.append(jc)
    return constraints

### 5-3. MotionPlanRequest 조립

In [11]:
from moveit_msgs.msg import MotionPlanRequest
def make_plan_request(joints: dict, vel: float, acc: float) -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    req.goal_constraints.append(make_joint_constraints(joints))
    return req

### 5-4. MoveGroup.Goal 감싸기

In [12]:
def make_goal(req: MotionPlanRequest) -> MoveGroup.Goal:
    goal = MoveGroup.Goal()
    goal.request = req
    return goal

### 5-5. 액션 송신과 결과 대기

리턴값은 `moveit_msgs/MoveItErrorCodes`의 정수값이다 (성공 시 `1`).

In [13]:
from moveit_msgs.msg import MoveItErrorCodes

def send_goal_and_wait(goal: MoveGroup.Goal) -> int:
    send_future = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED
    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    return result_future.result().result.error_code.val

### 5-6. 최종 진입점 — `go_to_joint_goal`

위 다섯 헬퍼를 순서대로 호출하기만 하는 얇은 오케스트레이터.

In [14]:
def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(joint_values, vel, acc)
    code = send_goal_and_wait(make_goal(req))
    ok = (code == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'MoveGroup 실패 error_code={code}')
    return ok

## 6. 시나리오 — 먼저 `home` 포즈로

In [15]:
node.get_logger().info('--- home 으로 초기 이동 ---')
go_to_joint_goal(home_target)
time.sleep(1.0)

[INFO] [1778198140.321730579] [ex03_joint_goal_demo]: --- home 으로 초기 이동 ---


## 7. 각 조인트 개별 이동

`step_joint(joint_name, angle, desc)` 한 번 호출 = `joint_name`만 `angle`로 보낸 뒤 다시 home으로 복귀.

In [16]:
import math

def step_joint(joint_name: str, angle: float, desc: str = '') -> None:
    deg = math.degrees(angle)
    node.get_logger().info(f'--- {joint_name} ({desc}) → {deg:.0f}° ---')
    target = {j: 0.0 for j in ARM_JOINTS}
    target[joint_name] = angle
    go_to_joint_goal(target, vel=0.3)
    time.sleep(1.5)
    node.get_logger().info('  → home 복귀')
    go_to_joint_goal(home_target, vel=0.3)
    time.sleep(1.0)

### joint1 — 베이스 회전

In [ ]:
step_joint('joint1', math.radians(45), '베이스 회전')

[INFO] [1778198175.649886222] [ex03_joint_goal_demo]: --- joint1 (베이스 회전) → 45° ---


### joint2 — 어깨 (앞뒤 기울기)

In [18]:
step_joint('joint2', math.radians(-30), '어깨 앞뒤 기울기')

[INFO] [1778063227.908925623] [ex03_joint_goal_demo]: --- joint2 (어깨 앞뒤 기울기) → -30° ---
[INFO] [1778063231.743914018] [ex03_joint_goal_demo]:   → home 복귀


### joint3 — 팔꿈치 (굽힘)

In [19]:
step_joint('joint3', math.radians(45), '팔꿈치 굽힘')

[INFO] [1778063109.671055316] [ex03_joint_goal_demo]: --- joint3 (팔꿈치 굽힘) → 45° ---
[INFO] [1778063114.363654281] [ex03_joint_goal_demo]:   → home 복귀


### joint4 — 손목 회전 (Roll)

In [ ]:
step_joint('joint4', math.radians(45), '손목 회전 Roll')

### joint5 — 손목 굽힘 (Pitch)

In [ ]:
step_joint('joint5', math.radians(30), '손목 굽힘 Pitch')

### joint6 — 손목 비틀기 (Yaw)

In [ ]:
step_joint('joint6', math.radians(45), '손목 비틀기 Yaw')

## 8. 보너스 — 여러 조인트 동시 이동

각 조인트가 서로 다른 목표값을 가질 때 MoveIt이 한 번에 다중 자유도를 보간해 보낸다.

In [ ]:
multi_target = {
    'joint1': math.radians( 30),
    'joint2': math.radians(-45),
    'joint3': math.radians( 60),
    'joint4': math.radians(  0),
    'joint5': math.radians( 45),
    'joint6': math.radians(-30),
}
node.get_logger().info('--- 여러 조인트 동시 이동 ---')
go_to_joint_goal(multi_target)
time.sleep(2.0)
go_to_joint_goal(home_target)
node.get_logger().info('=== 예제 03 완료! ===')

## 9. 정리

노트북을 닫기 전에 노드와 rclpy를 안전하게 정리한다.
다시 실행하려면 노트북 위쪽 셀부터 순서대로 다시 돌리면 된다.

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass